🧹**Data Preprocessing for Classifier Models**

-> URLs: URLs are removed.

-> User Mentions: Mentions (e.g., @username) are removed.

-> Hashtags: Hashtags (e.g., #hashtag) are removed.

-> Repetition: Repeated characters (like cooool -> cool) are reduced.

-> Convert Emojis & Emoticons: Emojis and emoticons are converted into readable text.

-> Digits & Punctuation: All numbers and punctuation are removed.

-> Extra Spaces: Multiple spaces are collapsed into a single space.

-> Trim White Spaces: Leading and trailing white spaces are removed.

-> Convert to Lowercase: All text is converted to lowercase.

-> Stopword Removal & Lemmatization: Stopwords are removed, and words are lemmatized to their root form.

In [2]:
from Lists.abbreviations import INTERNET_SLANG
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from Lists.emoticons import EMOTICONS
from spellchecker import SpellChecker
from Lists.emojis import EMO_UNICODE
from nltk.corpus import stopwords
from tqdm.notebook import tqdm
import pandas as pd
import contractions
import unicodedata
import nltk
import re

nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english')) - {'not', 'no', 'nor', 'never'}
lemmatizer = WordNetLemmatizer()
spell = SpellChecker()

tqdm.pandas()

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/bianca.guceanu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/bianca.guceanu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/bianca.guceanu/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [7]:
df = pd.read_csv('./dataset_loyola_university.csv')

df["Type"] = df["Type"].map({
    0: "not_cyberbullying",
    1: "cyberbullying"
})

df = df.rename(columns={
    'Comment': 'text',
    'Type': 'type'
})

df.head()

,text,type,"1= Sexual; but NOT sexual/gender identity (e.g. ""whore"") 0= N/A","1= Sexual orientation/gender identity (e.g. ""gay"") 0= N/A",1 = Physical Appearance 0= N/A,1= Race/Ethnicity 0= N/A,1= Intellectual 0= N/A,1= Religious 0= N/A,"1 = General Hate (F* Off, Go Die, etc) 0= N/A",1= Insult /Attack 0= N/A,1= Defense of Another 2= Defense of Self 0= N/A,1= Depression 0= N/A,1= Suicide 0= N/A,1= Stress (or anxiety) 0= N/A,1= Discrimination 0= N/A
0,Zaaaayn,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Larry,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Niall_Å½Ã›_,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,How do u get a gif? I cant aave them to my phone,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,"Larry, Zayn being sexy, and Niall and Liam doi...",not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
type_labels = {label: idx for idx, label in enumerate(df['type'].unique())}
print(type_labels)
df['label'] = df['type'].map(type_labels)

{'not_cyberbullying': 0, 'cyberbullying': 1}


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8470 entries, 0 to 8478
Data columns (total 16 columns):
 #   Column                                                           Non-Null Count  Dtype  
---  ------                                                           --------------  -----  
 0   text                                                             8470 non-null   object 
 1   type                                                             8470 non-null   object 
 2   1= Sexual; but NOT sexual/gender identity (e.g. "whore") 0= N/A  8470 non-null   object 
 3   1= Sexual orientation/gender identity (e.g. "gay") 0= N/A        8470 non-null   float64
 4   1 = Physical Appearance 0= N/A                                   8470 non-null   float64
 5   1= Race/Ethnicity 0= N/A                                         8470 non-null   object 
 6   1= Intellectual 0= N/A                                           8470 non-null   float64
 7   1= Religious 0= N/A                            

In [16]:
df.head()

,text,type,"1= Sexual; but NOT sexual/gender identity (e.g. ""whore"") 0= N/A","1= Sexual orientation/gender identity (e.g. ""gay"") 0= N/A",1 = Physical Appearance 0= N/A,1= Race/Ethnicity 0= N/A,1= Intellectual 0= N/A,1= Religious 0= N/A,"1 = General Hate (F* Off, Go Die, etc) 0= N/A",1= Insult /Attack 0= N/A,1= Defense of Another 2= Defense of Self 0= N/A,1= Depression 0= N/A,1= Suicide 0= N/A,1= Stress (or anxiety) 0= N/A,1= Discrimination 0= N/A,label
0,Zaaaayn,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,Larry,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,Niall_Å½Ã›_,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,How do u get a gif? I cant aave them to my phone,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,"Larry, Zayn being sexy, and Niall and Liam doi...",not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [17]:
UNICODE_EMO = {v: k for k, v in EMO_UNICODE.items()}

def convert_emoticons(text):
    for emot in EMOTICONS:
        text = re.sub(u'(' + emot + ')', "  ".join(EMOTICONS[emot].replace(",", "").split()), text)
    return text

def convert_emojis(text):
    for emot in UNICODE_EMO:
        text = re.sub(r'(' + emot + ')', "  ".join(UNICODE_EMO[emot].replace(",", "").replace(":", "").split()), text)
    return text

def convert_abbreviations(text):
    for abbr in INTERNET_SLANG:
        pattern = re.compile(r'\b' + re.escape(abbr) + r'\b', re.IGNORECASE)
        replacement = " ".join(INTERNET_SLANG[abbr].replace(",", "").split())
        text = pattern.sub(replacement, text)
    return text

def remove_unknown_emojis(text):
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"  # miscellaneous symbols
        u"\U0001F900-\U0001F9FF"  # supplemental symbols
        u"\U0001F200-\U0001F251"  # enclosed characters
        u"\U0001F004-\U0001F0CF"  # playing cards
        u"\U00002B50"  # star symbol
        "]+", flags=re.UNICODE)

    emojis_in_text = emoji_pattern.findall(text)

    for emoji_char in emojis_in_text:
        if emoji_char not in UNICODE_EMO:
            text = text.replace(emoji_char, '')
    return text

def remove_accented_chars(text):
    text = contractions.fix(text)
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

In [18]:
def convert_emojis_and_slang(text):
    text = convert_emoticons(text)
    text = convert_emojis(text)
    text = remove_unknown_emojis(text)
    text = convert_abbreviations(text)
    text=remove_accented_chars(text)

    return text

In [19]:
 def correction_stopwords_lemmatize(text):
        if not isinstance(text, str):
            return ""

        text = contractions.fix(text)
        tokens = word_tokenize(text)
        corrected_and_lemmatized_tokens = []

        for word in tokens:
            if word in stop_words:
                continue

            corrected = word
            if spell.unknown([word]):
                possible_correction = spell.correction(word)
                if possible_correction is not None:
                    corrected = possible_correction

            lemmatized = lemmatizer.lemmatize(corrected)
            corrected_and_lemmatized_tokens.append(lemmatized)

        return " ".join(corrected_and_lemmatized_tokens)

In [20]:
sequencePattern = r"(.)\1\1+"
seqReplacePattern = r"\1\1"

In [21]:
def preprocess_text(df_input):
    df_preprocessing=df_input.copy()
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r"http\S+", "URL", regex=True)  # Remove URLs
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'@[A-Za-z0-9_.]+', 'USER', regex=True)  # Remove user mentions
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'#+(\S+)', r'\1', regex=True)  # Remove hashtags
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(sequencePattern, seqReplacePattern, regex=True)  # Remove repetition

    df_preprocessing['text'] = df_preprocessing['text'].progress_apply(lambda x: convert_emojis_and_slang(x))
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'\d+', '', regex=True)  # Remove digits
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'[^\w\s]', '', regex=True)  # Remove punctuation
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'\s+', ' ', regex=True)  # Remove spaces
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'[^a-zA-Z\s]', '', regex=True)  # Only words
    df_preprocessing['text'] = df_preprocessing['text'].str.replace(r'\s+', ' ', regex=True).str.strip()  # Trim spaces
    df_preprocessing['text'] = df_preprocessing['text'].str.lower()  # Convert text to lowercase
    df_preprocessing['text'] = df_preprocessing['text'].progress_apply(lambda x: correction_stopwords_lemmatize(x))

    df_preprocessing = df_preprocessing.dropna(how='any')
    df_preprocessing = df_preprocessing.drop_duplicates()

    return df_preprocessing

In [22]:
df = preprocess_text(df)

  0%|          | 0/8470 [00:00<?, ?it/s]

  0%|          | 0/8470 [00:00<?, ?it/s]

In [23]:
df.head()

,text,type,"1= Sexual; but NOT sexual/gender identity (e.g. ""whore"") 0= N/A","1= Sexual orientation/gender identity (e.g. ""gay"") 0= N/A",1 = Physical Appearance 0= N/A,1= Race/Ethnicity 0= N/A,1= Intellectual 0= N/A,1= Religious 0= N/A,"1 = General Hate (F* Off, Go Die, etc) 0= N/A",1= Insult /Attack 0= N/A,1= Defense of Another 2= Defense of Self 0= N/A,1= Depression 0= N/A,1= Suicide 0= N/A,1= Stress (or anxiety) 0= N/A,1= Discrimination 0= N/A,label
0,zany,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,larry,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,niallaa,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,get if not have phone,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,larry zany sexy all liam something stupid back,not_cyberbullying,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6479 entries, 0 to 8478
Data columns (total 16 columns):
 #   Column                                                           Non-Null Count  Dtype  
---  ------                                                           --------------  -----  
 0   text                                                             6479 non-null   object 
 1   type                                                             6479 non-null   object 
 2   1= Sexual; but NOT sexual/gender identity (e.g. "whore") 0= N/A  6479 non-null   object 
 3   1= Sexual orientation/gender identity (e.g. "gay") 0= N/A        6479 non-null   float64
 4   1 = Physical Appearance 0= N/A                                   6479 non-null   float64
 5   1= Race/Ethnicity 0= N/A                                         6479 non-null   object 
 6   1= Intellectual 0= N/A                                           6479 non-null   float64
 7   1= Religious 0= N/A                            

In [25]:
df.to_csv('./dataset_loyola_preprocessed.csv',index=False)